## Notebook 4: Model Right-Sizing & Cross-Benchmarking

    In this notebook, we will evaluate the extractor using various local open-source models. The goal is to identify the best model while taking performance and latency into account. We will use the sample prompts created in Notebook 1, the system prompt and extraction schema created in Notebook 2, and the evaluator described in Notebook 3. The results will be saved to a CSV file for further analysis.


### Notebook Workflow
1. Start the Ollama server in the terminal using the command “ollama serve.”<br> Ask your chatbot assistant for recommendations about ollama models that are suitable for your local computer’s CPU, RAM, and GPU.<br>Download the selected set of open-source models from Ollama.<br><br>
2. Add the models to the “model_list” in the benchmarking loop. <br>The script uses the basic “extractor” function, <br>passes the results to the extractor, <br>and saves all results to the same CSV file.<br>The script also measures and records the latency for each extraction (pizza level, order level, and dataset level).<br><br>
3. Compare and visualize the benchmark results using Pandas.
- 3.0 Load and prepare dataset for visualization
- 3.1 Global Accuracy Ranking: Overall Micro-Jaccard score across the entire dataset.
- 3.2  Speed vs. Accuracy Efficiency Matrix: Scatter plot identifying models in the high-speed/high-accuracy optimal zone.
- 3.3 Field-Level Precision Heatmap: Pinpoints schema fields where specific models underperform (e.g., complex nested list fields vs. primitive scalars).
- 3.4 Latency Distribution: Boxplot illustrating latency variance and response stability per sample.










### 1. The selected open source LLMs from ollama or propietary LLMs
Insert a dict with the keys `"model"` and `"provider"` for each LLM.<br><br>
Example: <br>
```python
[
    {"model": "llama3.1:8b", "provider": "ollama"},

    {"model": "claude-haiku-5-5", "provider": "anthropic"}
]
```


In [ ]:
models_config = [
    # Ollama Models

    {"model": "llama3.1:8b", "provider": "ollama"},

    {"model": "mistral:7b", "provider": "ollama"},
    {"model": "mistral-nemo:12b", "provider": "ollama"},

    {"model": "qwen2.5:7b-instruct", "provider": "ollama"},
    {"model": "qwen3:4b-instruct", "provider": "ollama"},
    {"model": "qwen3.5:9b", "provider": "ollama"},
    
    # Anthropic Model
    {"model": "claude-haiku-4-5-20251001", "provider": "anthropic"},
    {"model": "claude-haiku-5-5", "provider": "anthropic"}
]

### 2.1 The Extractor Function (`run_extractor`)
This function receives the `current model`, loops through the `testorders` and returns a response_list with the `extraction results` and the `extraction latency`.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from pizza_order_extractor.prompts import PIZZA_SYSTEM_PROMPT
from pizza_order_extractor.schemas import PizzaOrder
import time
from pizza_order_extractor.helper_functions import check_apikeys

check_apikeys()

def run_extractor(model_name, provider, test_orders):
    model = init_chat_model(model_name, model_provider=provider)
    
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", PIZZA_SYSTEM_PROMPT),
        ("user", "{order_text}"),
    ])
    
    structured_llm = model.with_structured_output(schema=PizzaOrder, method="json_schema")
    extractor = prompt_template | structured_llm
    
    
    response_list = []
    duration_list = []

    for i, order in enumerate(test_orders, 1):

        # Track wall-clock duration
        start_time = time.perf_counter()
        response = extractor.invoke({"order_text": order})
        end_time = time.perf_counter()
                
        duration = round(end_time - start_time, 3)
        duration_list.append(duration)
    
        response_list.append(response)
        print(f"--- Order {i} ---\n{order}")
        print(f"--- Extracted Output {i} ---\n{response}\n")
        
    return response_list, duration_list 

### 2.2 The Extraction and Evaluation Loop
This script loops through the model list and 
- calls the run_extractor function we defined before
- saves the result to /extraction_results
- sends the extraction results and the Golden Dataset to the evaluator
- returns a pandas spreadsheet with the detailed results for each LLM.




In [ ]:
import pandas as pd
from pathlib import Path
from pizza_order_extractor.helper_functions import save_extraction_to_file, load_orders_as_list
from pizza_order_extractor.evaluator import load_and_evaluate


# 1. Define model configurations

order_file = "pedidos_pizza_20260916_200028.json"
test_orders = load_orders_as_list(order_file)

# 2. Evaluation loop
for config in models_config:
    model_name = config["model"]
    provider = config["provider"]

    
    try:
        
        response_list, duration_list = run_extractor(model_name, provider, test_orders)
        
        _ , filename = save_extraction_to_file(response_list, model_name)

        df = load_and_evaluate("golden_dataset.json", filename, duration_list)
        
    except Exception as e:
        print(f"Error executing {model_name}: {e}")


df.head(10)

In [ ]:
# a first general overview 
df [df ["level"]== "dataset_summary"].iloc[:, [0,4,15]]


### 3.0 Load and prepare dataset for visualization

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure global plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.sans-serif": "DejaVu Sans",
    "font.size": 10,
    "figure.titlesize": 14
})

# Path to benchmark results CSV
CSV_PATH = "data/evaluation_results/extractor_benchmark_results.csv"

# Load data
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

# Extract clean model name (e.g. "extraction_gpt-4o.json" -> "gpt-4o")
df["model"] = (
    df["dataset_name"]
    .astype(str)
    .str.replace(r"^extraction_", "", regex=True)
    .str.replace(r"_\d{8}_\d{6}", "", regex=True)  # Strips _20261030_155011 pattern
    .str.replace(r"\.json$", "", regex=True)
)

# Identify scalar and nested field columns dynamically
metadata_cols = {"dataset_name", "sample_id", "level", "item_id", "duration_sec", "jaccard_score", "model"}
schema_fields = [col for col in df.columns if col not in metadata_cols]

print(f"Loaded {len(df)} total rows across models: {df['model'].unique().tolist()}")
print(f"Detected {len(schema_fields)} schema fields: {schema_fields}")

### 3.1 Global Accuracy Ranking: Overall Micro-Jaccard score across the entire dataset.

In [ ]:
# Filter dataset level summary rows
ds_df = df[df["level"] == "dataset_summary"].copy()
ds_df = ds_df.sort_values(by="jaccard_score", ascending=False)

plt.figure(figsize=(10, 5))
bars = sns.barplot(
    data=ds_df, 
    x="model", 
    y="jaccard_score", 
    palette="Blues_r"
)

plt.title("Overall Global Dataset Accuracy (Micro-Jaccard Score)", fontsize=13, fontweight="bold")
plt.ylabel("Global Jaccard Score (0.0 - 1.0)")
plt.xlabel("LLM Model")
plt.ylim(0, 1.1)

# Annotate exact percentage values on top of bars
for p in bars.patches:
    height = p.get_height()
    if not pd.isna(height):
        bars.annotate(
            f"{height * 100:.1f}%",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center", va="center", xytext=(0, 8),
            textcoords="offset points", fontweight="bold"
        )

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 3.2  Speed vs. Accuracy Efficiency Matrix: Scatter plot identifying models in the high-speed/high-accuracy optimal zone.

In [ ]:
# Filter sample level summary rows
sample_df = df[df["level"] == "sample_summary"].copy()

# Compute mean latency and accuracy per model
model_stats = sample_df.groupby("model").agg(
    avg_latency=("duration_sec", "mean"),
    avg_accuracy=("jaccard_score", "mean")
).reset_index()

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=model_stats, 
    x="avg_latency", 
    y="avg_accuracy",
    hue="model", 
    style="model", 
    s=250, 
    markers=True
)

plt.title("Accuracy vs. Speed (Efficiency Matrix)", fontsize=13, fontweight="bold")
plt.xlabel("Average Latency per Order (Seconds)")
plt.ylabel("Average Sample Jaccard Score")

# Plot median quadrant dividers
med_time = model_stats["avg_latency"].median()
med_acc = model_stats["avg_accuracy"].median()
plt.axvline(med_time, color="gray", linestyle="--", alpha=0.5)
plt.axhline(med_acc, color="gray", linestyle="--", alpha=0.5)

# Label optimal quadrant
plt.text(
    model_stats["avg_latency"].min(), 
    model_stats["avg_accuracy"].max(), 
    "High Speed & High Accuracy\n(Optimal Zone)", 
    color="green", fontweight="bold", alpha=0.75
)

# Label data points
for _, row in model_stats.iterrows():
    plt.annotate(
        row["model"], 
        (row["avg_latency"], row["avg_accuracy"]),
        textcoords="offset points", 
        xytext=(8, 5), 
        ha="left", 
        fontweight="semibold"
    )

plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 3.3 Field-Level Precision Heatmap: Pinpoints schema fields where specific models underperform (e.g., complex nested list fields vs. primitive scalars).

In [ ]:
# Filter item level rows
item_df = df[df["level"] == "item"].copy()

# Compute mean accuracy per schema attribute
field_acc = item_df.groupby("model")[schema_fields].mean()

plt.figure(figsize=(11, 6))
sns.heatmap(
    field_acc, 
    annot=True, 
    fmt=".2f", 
    cmap="YlGnBu", 
    cbar=True, 
    vmin=0.5, 
    vmax=1.0,
    linewidths=0.5
)

plt.title("Field-Level Accuracy Breakdown Heatmap", fontsize=13, fontweight="bold")
plt.ylabel("LLM Model")
plt.xlabel("Schema Attributes")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 3.4 Latency Distribution: Boxplot illustrating latency variance and response stability per sample.

In [ ]:
# Filter sample level summary rows
sample_df = df[df["level"] == "sample_summary"].dropna(subset=["duration_sec"]).copy()

plt.figure(figsize=(10, 5))

# Plot distribution boxes
sns.boxplot(
    data=sample_df, 
    x="model", 
    y="duration_sec", 
    palette="Set3",
    showfliers=False
)

# Overlay individual sample points
sns.stripplot(
    data=sample_df, 
    x="model", 
    y="duration_sec", 
    color="black", 
    alpha=0.4, 
    jitter=0.2,
    size=5
)

plt.title("Latency Distribution per Order (Seconds)", fontsize=13, fontweight="bold")
plt.ylabel("Duration (Seconds)")
plt.xlabel("LLM Model")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Congrats. You reached the end of this tutorial. 
We hope you enjoyed your IDP journey up to this point. 

Here is where the real work begins: 
- Identify a good llm candidate for production. 
- Iterate through the schema definition to optimize the performance for this model. 
- Check for new models that could perform better ...